In [1]:
import cv2
import numpy as np
import os

# Folders
input_folder = "road_member3_sharpened"
output_folder = "output_member2"

os.makedirs(output_folder, exist_ok=True)

for i in range(17, 31):

    
    # INPUT FRAME
    
    img_path = os.path.join(input_folder, f"gaussian_contrast_frame_{i}.jpg")
    image = cv2.imread(img_path)

    if image is None:
        print(f"Frame {i} not found")
        continue

    cv2.imwrite(f"{output_folder}/{i}_1_input.jpg", image)

    
    # PREPROCESSING (GRAY + BLUR)
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    cv2.imwrite(f"{output_folder}/{i}_2_gray.jpg", gray)
    cv2.imwrite(f"{output_folder}/{i}_3_blur.jpg", blur)

    
    # SEGMENTATION (BLACKHAT + THRESHOLD)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15,15))
    blackhat = cv2.morphologyEx(blur, cv2.MORPH_BLACKHAT, kernel)

    norm = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)

    _, thresh = cv2.threshold(norm, 30, 255, cv2.THRESH_BINARY)

    cv2.imwrite(f"{output_folder}/{i}_4_blackhat.jpg", blackhat)
    cv2.imwrite(f"{output_folder}/{i}_5_thresh.jpg", thresh)

    
    # MORPHOLOGICAL CLEANING
    
    kernel_small = np.ones((3,3), np.uint8)

    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_small)
    dilation = cv2.dilate(opening, kernel_small, iterations=2)

    cv2.imwrite(f"{output_folder}/{i}_6_opening.jpg", opening)
    cv2.imwrite(f"{output_folder}/{i}_7_dilation.jpg", dilation)

    
    # EDGE DETECTION
    
    edges = cv2.Canny(dilation, 50, 150)

    cv2.imwrite(f"{output_folder}/{i}_8_edges.jpg", edges)


    # CONTOUR DETECTION + FILTERING
    
    contours, _ = cv2.findContours(dilation, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    output = image.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area > 200:
            x, y, w, h = cv2.boundingRect(cnt)

            roi = dilation[y:y+h, x:x+w]
            density = cv2.countNonZero(roi) / (w * h)

            if 0.15 < density < 0.5:
                cv2.rectangle(output, (x,y), (x+w,y+h), (0,255,0), 2)
                cv2.putText(output, "Alligator Crack", (x, y-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    cv2.imwrite(f"{output_folder}/{i}_9_contours.jpg", output)

print("Detection Completed!")

Detection Completed!
